In [1]:
!pip install -q timm detectors datasets huggingface_hub


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import copy
import random
import numpy as np
import matplotlib.pyplot as plt

import timm
import detectors

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import DataLoader, Dataset
from torchvision import datasets, transforms, models
from tqdm import tqdm
from datasets import load_dataset

d:\Python\Juicer-for-models\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
d:\Python\Juicer-for-models\venv\Lib\site-packages\outdated\__init__.py:36: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import parse_version


In [3]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Устройство:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Устройство: cuda
GPU: Tesla T4


### Загрузка датасета и создание лоадеров

In [ ]:
CIFAR10_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR10_STD = (0.2470, 0.2435, 0.2616)

class HFCifar10Dataset(Dataset):
    def __init__(self, hf_dataset, transform=None):
        self.hf_dataset = hf_dataset
        self.transform = transform
        self.classes = hf_dataset.features["label"].names

    def __len__(self):
        return len(self.hf_dataset)

    def __getitem__(self, index):
        item = self.hf_dataset[index]

        image = item["img"]
        label = item["label"]

        if self.transform is not None:
            image = self.transform(image)

        label = torch.tensor(label, dtype=torch.long)

        return image, label

train_transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize(CIFAR10_MEAN, CIFAR10_STD)])
test_transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize(CIFAR10_MEAN, CIFAR10_STD),])

In [ ]:
train_data = load_dataset(
    "uoft-cs/cifar10",
    split="train",
    cache_dir="../data/processed/hf_cache"
)

test_data = load_dataset(
    "uoft-cs/cifar10",
    split="test",
    cache_dir="../data/processed/hf_cache"
)

train_dataset = HFCifar10Dataset(train_data, transform=train_transform)
test_dataset = HFCifar10Dataset(test_data, transform=test_transform)

In [6]:
BATCH_SIZE = 256

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=torch.cuda.is_available(),
    persistent_workers=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=torch.cuda.is_available(),
    persistent_workers=True
)

print("Обучающих изображений:", len(train_dataset))
print("Тестовых изображений:", len(test_dataset))
print("Количество классов:", len(train_dataset.classes))
print("Классы:", train_dataset.classes)

Обучающих изображений: 50000
Тестовых изображений: 10000
Количество классов: 10
Классы: ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']


### Загрузка предобученной ResNet50 и проверка

In [ ]:
teacher = detectors.create_model("resnet50_cifar10", pretrained=True)

teacher = teacher.to(device)
teacher.eval()

for parameter in teacher.parameters():
    parameter.requires_grad = False

### Создание ResNet18

In [ ]:
def create_cifar_resnet18(num_classes=10):
    model = models.resnet18(weights=None)

    # Адаптация под изображения CIFAR-10 размером 32×32
    model.conv1 = nn.Conv2d(
        in_channels=3,
        out_channels=64,
        kernel_size=3,
        stride=1,
        padding=1,
        bias=False
    )

    model.maxpool = nn.Identity()
    model.fc = nn.Linear(model.fc.in_features, num_classes)

    return model

In [10]:
student = create_cifar_resnet18().to(device)

### Класс получения промежуточных карт признаков

In [ ]:
class FeatureExtractor:
    def __init__(self, model, layer_names):
        self.features = {}
        self.handles = []

        modules = dict(model.named_modules())

        for layer_name in layer_names:
            if layer_name not in modules:
                raise ValueError(f"Слой {layer_name!r} не найден в модели")

            handle = modules[layer_name].register_forward_hook(self._create_hook(layer_name))

            self.handles.append(handle)

    def _create_hook(self, layer_name):
        def hook(module, inputs, output):
            self.features[layer_name] = output

        return hook

    def clear(self):
        self.features.clear()

    def remove(self):
        for handle in self.handles:
            handle.remove()

        self.handles.clear()

In [ ]:
DISTILLATION_LAYERS = ["layer1", "layer2", "layer3", "layer4"]

teacher_extractor = FeatureExtractor(teacher, DISTILLATION_LAYERS)
student_extractor = FeatureExtractor(student, DISTILLATION_LAYERS)

In [13]:
images, labels = next(iter(train_loader))

images = images.to(device)

teacher_extractor.clear()
student_extractor.clear()

with torch.no_grad():
    teacher_logits = teacher(images)

student_logits = student(images)

print("Teacher logits:", teacher_logits.shape)
print("Student logits:", student_logits.shape)

for layer_name in DISTILLATION_LAYERS:
    student_shape = student_extractor.features[layer_name].shape
    teacher_shape = teacher_extractor.features[layer_name].shape

    print(
        f"{layer_name}: "
        f"student={tuple(student_shape)}, "
        f"teacher={tuple(teacher_shape)}"
    )

Teacher logits: torch.Size([256, 10])
Student logits: torch.Size([256, 10])
layer1: student=(256, 64, 32, 32), teacher=(256, 256, 32, 32)
layer2: student=(256, 128, 16, 16), teacher=(256, 512, 16, 16)
layer3: student=(256, 256, 8, 8), teacher=(256, 1024, 8, 8)
layer4: student=(256, 512, 4, 4), teacher=(256, 2048, 4, 4)


### Класс адептера карт признаков

In [ ]:
class FeatureAdapters(nn.Module):
    def __init__(self):
        super().__init__()

        self.adapters = nn.ModuleDict({
            "layer1": nn.Conv2d(64, 256, kernel_size=1, bias=False),
            "layer2": nn.Conv2d(128, 512, kernel_size=1, bias=False),
            "layer3": nn.Conv2d(256, 1024, kernel_size=1, bias=False),
            "layer4": nn.Conv2d(512, 2048, kernel_size=1, bias=False),
        })

    def forward(self, features):
        return {
            layer_name: self.adapters[layer_name](
                features[layer_name]
            )
            for layer_name in self.adapters
        }

In [15]:
feature_adapters = FeatureAdapters().to(device)

### Функция потерь по картам признаков - MSE

In [ ]:
def normalize_feature_map(feature_map, eps=1e-6):
    """
    Нормализация каждого объекта и каждого канала
    по пространственным координатам H×W.
    """
    mean = feature_map.mean(dim=(2, 3), keepdim=True)
    std = feature_map.std(dim=(2, 3), keepdim=True)

    return (feature_map - mean) / (std + eps)

def feature_distillation_loss(
    student_features,
    teacher_features,
    adapters,
    layer_weights=None
):
    if layer_weights is None:
        layer_weights = {
            "layer1": 1.0,
            "layer2": 1.0,
            "layer3": 1.0,
            "layer4": 1.0,
        }

    adapted_student_features = adapters(student_features)
    total_feature_loss = torch.zeros((), device=next(adapters.parameters()).device)

    layer_losses = {}

    for layer_name, student_feature in (adapted_student_features.items()):
        teacher_feature = teacher_features[layer_name].detach()

        if student_feature.shape[2:] != teacher_feature.shape[2:]:
            student_feature = F.interpolate(
                student_feature,
                size=teacher_feature.shape[2:],
                mode="bilinear",
                align_corners=False
            )

        normalized_student = normalize_feature_map(student_feature)
        normalized_teacher = normalize_feature_map(teacher_feature)
        layer_loss = F.mse_loss(normalized_student, normalized_teacher)

        layer_losses[layer_name] = layer_loss

        total_feature_loss = (total_feature_loss + layer_weights[layer_name] * layer_loss)

    weight_sum = sum(layer_weights.values())
    total_feature_loss = total_feature_loss / weight_sum

    return total_feature_loss, layer_losses

### Полная функция дистилляции

In [ ]:
def combined_distillation_loss(
    student_logits,
    teacher_logits,
    labels,
    student_features,
    teacher_features,
    adapters,
    temperature=4.0,
    ce_weight=1.0,
    logits_weight=1.0,
    feature_weight=1.0,
    layer_weights=None
):
    ce_loss = F.cross_entropy(student_logits, labels)
    student_log_probs = F.log_softmax(student_logits / temperature, dim=1)
    teacher_probs = F.softmax(teacher_logits.detach() / temperature, dim=1)

    logits_kd_loss = F.kl_div(
        student_log_probs,
        teacher_probs,
        reduction="batchmean"
    ) * temperature**2

    feature_loss, layer_losses = (
        feature_distillation_loss(
            student_features=student_features,
            teacher_features=teacher_features,
            adapters=adapters,
            layer_weights=layer_weights
        )
    )

    total_loss = (ce_weight * ce_loss + logits_weight * logits_kd_loss + feature_weight * feature_loss)

    return {
        "total": total_loss,
        "ce": ce_loss,
        "logits_kd": logits_kd_loss,
        "feature": feature_loss,
        "layers": layer_losses
    }

### Оптимизатор

In [ ]:
EPOCHS = 10

optimizer = torch.optim.AdamW(
    list(student.parameters()) +
    list(feature_adapters.parameters()),
    lr=1e-3,
    weight_decay=1e-4,
    betas=(0.9, 0.999),
    eps=1e-8
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
scaler = torch.amp.GradScaler("cuda", enabled=device.type == "cuda")

### Функция оценки

In [ ]:
@torch.no_grad()
def evaluate_student(model, data_loader):
    model.eval()

    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    for images, labels in data_loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        logits = model(images)

        loss = F.cross_entropy(logits, labels)

        batch_size = labels.size(0)

        total_loss += loss.item() * batch_size
        total_correct += (logits.argmax(dim=1) == labels).sum().item()

        total_samples += batch_size

    return (
        total_loss / total_samples,
        total_correct / total_samples
    )

### Обучение

In [20]:
best_accuracy = 0.0
best_student_state = None

history = {
    "total_loss": [],
    "ce_loss": [],
    "logits_kd_loss": [],
    "feature_loss": [],
    "train_accuracy": [],
    "test_accuracy": [],
}

In [ ]:
for epoch in range(EPOCHS):
    student.train()
    feature_adapters.train()
    teacher.eval()

    total_samples = 0
    total_correct = 0

    running_total_loss = 0.0
    running_ce_loss = 0.0
    running_logits_loss = 0.0
    running_feature_loss = 0.0

    running_layer_losses = {layer_name: 0.0 for layer_name in DISTILLATION_LAYERS}

    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch + 1}/{EPOCHS}")

    for images, labels in progress_bar:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        batch_size = labels.size(0)

        optimizer.zero_grad(set_to_none=True)

        teacher_extractor.clear()
        student_extractor.clear()

        with torch.no_grad():
            with torch.autocast(
                device_type=device.type,
                enabled=device.type == "cuda"
            ):
                teacher_logits = teacher(images)

        # Forward ученика
        with torch.autocast(device_type=device.type, enabled=device.type == "cuda"):
            student_logits = student(images)

            losses = combined_distillation_loss(
                student_logits=student_logits,
                teacher_logits=teacher_logits,
                labels=labels,
                student_features=student_extractor.features,
                teacher_features=teacher_extractor.features,
                adapters=feature_adapters
            )

            loss = losses["total"]

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        predictions = student_logits.argmax(dim=1)

        total_correct += (predictions == labels).sum().item()
        total_samples += batch_size

        running_total_loss += (losses["total"].item() * batch_size)
        running_ce_loss += (losses["ce"].item() * batch_size)
        running_logits_loss += (losses["logits_kd"].item() * batch_size)
        running_feature_loss += (losses["feature"].item() * batch_size)

        for layer_name, layer_loss in (losses["layers"].items()):
            running_layer_losses[layer_name] += (
                layer_loss.item() * batch_size
            )

        progress_bar.set_postfix({
            "loss": f"{loss.item():.3f}",
            "CE": f"{losses['ce'].item():.3f}",
            "KD": f"{losses['logits_kd'].item():.3f}",
            "feat": f"{losses['feature'].item():.3f}",
        })

    scheduler.step()

    train_total_loss = (running_total_loss / total_samples)
    train_ce_loss = (running_ce_loss / total_samples)
    train_logits_loss = (running_logits_loss / total_samples)
    train_feature_loss = (running_feature_loss / total_samples)

    train_accuracy = (total_correct / total_samples)

    test_loss, test_accuracy = evaluate_student(student, test_loader)

    history["total_loss"].append(train_total_loss)
    history["ce_loss"].append(train_ce_loss)
    history["logits_kd_loss"].append(train_logits_loss)
    history["feature_loss"].append(train_feature_loss)
    history["train_accuracy"].append(train_accuracy)
    history["test_accuracy"].append(test_accuracy)

    if test_accuracy > best_accuracy:
        best_accuracy = test_accuracy

        best_student_state = copy.deepcopy(student.state_dict())

    layer_text = " | ".join(
        f"{layer_name}="
        f"{running_layer_losses[layer_name] / total_samples:.3f}"
        for layer_name in DISTILLATION_LAYERS
    )

    print(
        f"Epoch {epoch + 1:02d}/{EPOCHS} | "
        f"total={train_total_loss:.4f} | "
        f"CE={train_ce_loss:.4f} | "
        f"logits={train_logits_loss:.4f} | "
        f"features={train_feature_loss:.4f} | "
        f"train acc={train_accuracy * 100:.2f}% | "
        f"test acc={test_accuracy * 100:.2f}%"
    )

    print(layer_text)

Epoch 1/10: 100%|██████████| 196/196 [00:56<00:00,  3.46it/s, loss=9.616, CE=1.166, KD=7.398, feat=1.052]


Epoch 01/10 | total=12.5391 | CE=1.5521 | logits=9.7149 | features=1.2721 | train acc=56.11% | test acc=63.07%
layer1=1.151 | layer2=1.295 | layer3=1.419 | layer4=1.223


Epoch 2/10: 100%|██████████| 196/196 [00:56<00:00,  3.49it/s, loss=7.492, CE=1.164, KD=5.407, feat=0.920]


Epoch 02/10 | total=7.4665 | CE=0.9683 | logits=5.5241 | features=0.9741 | train acc=75.49% | test acc=76.62%
layer1=0.758 | layer2=0.964 | layer3=1.166 | layer4=1.008


Epoch 3/10: 100%|██████████| 196/196 [00:57<00:00,  3.39it/s, loss=5.837, CE=0.694, KD=4.261, feat=0.881]


...

Epoch 08/10 | total=1.1785 | CE=0.0058 | logits=0.3797 | features=0.7930 | train acc=99.86% | test acc=87.65%
layer1=0.586 | layer2=0.794 | layer3=1.022 | layer4=0.769


Epoch 9/10: 100%|██████████| 196/196 [01:00<00:00,  3.24it/s, loss=1.167, CE=0.003, KD=0.373, feat=0.791]


Epoch 09/10 | total=1.1138 | CE=0.0031 | logits=0.3242 | features=0.7865 | train acc=99.93% | test acc=88.08%
layer1=0.581 | layer2=0.789 | layer3=1.017 | layer4=0.760


Epoch 10/10: 100%|██████████| 196/196 [01:00<00:00,  3.23it/s, loss=1.060, CE=0.001, KD=0.274, feat=0.785]


Epoch 10/10 | total=1.0873 | CE=0.0022 | logits=0.3014 | features=0.7838 | train acc=99.97% | test acc=87.91%
layer1=0.579 | layer2=0.786 | layer3=1.015 | layer4=0.756


### Обучение ResNet18 без дистилляции

In [22]:
student_plain = create_cifar_resnet18().to(device)

In [ ]:
EPOCHS = 10

optimizer_plain = torch.optim.AdamW(
    student_plain.parameters(),
    lr=1e-3,
    weight_decay=1e-4,
    betas=(0.9, 0.999),
    eps=1e-8
)

scheduler_plain = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer_plain, T_max=EPOCHS)
scaler_plain = torch.amp.GradScaler("cuda", enabled=device.type == "cuda")

In [24]:
best_plain_accuracy = 0.0
best_plain_state = None

plain_history = {
    "train_loss": [],
    "train_accuracy": [],
    "test_loss": [],
    "test_accuracy": []
}

In [ ]:
for epoch in range(EPOCHS):
    student_plain.train()

    running_loss = 0.0
    total_correct = 0
    total_samples = 0

    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch + 1}/{EPOCHS}")

    for images, labels in progress_bar:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer_plain.zero_grad(set_to_none=True)

        with torch.autocast(device_type=device.type, enabled=device.type == "cuda"):
            logits = student_plain(images)

            loss = F.cross_entropy(logits, labels)

        scaler_plain.scale(loss).backward()
        scaler_plain.step(optimizer_plain)
        scaler_plain.update()

        batch_size = labels.size(0)

        running_loss += (loss.item() * batch_size)
        predictions = logits.argmax(dim=1)

        total_correct += (predictions == labels).sum().item()
        total_samples += batch_size

        progress_bar.set_postfix({"loss": f"{loss.item():.4f}"})

    scheduler_plain.step()

    train_loss = running_loss / total_samples
    train_accuracy = total_correct / total_samples

    test_loss, test_accuracy = evaluate_student(student_plain, test_loader)

    plain_history["train_loss"].append(train_loss)
    plain_history["train_accuracy"].append(train_accuracy)
    plain_history["test_loss"].append(test_loss)
    plain_history["test_accuracy"].append(test_accuracy)

    if test_accuracy > best_plain_accuracy:
        best_plain_accuracy = test_accuracy

        best_plain_state = copy.deepcopy(
            student_plain.state_dict()
        )

    current_lr = optimizer_plain.param_groups[0]["lr"]

    print(
        f"Epoch {epoch + 1:02d}/{EPOCHS} | "
        f"lr={current_lr:.6f} | "
        f"train loss={train_loss:.4f} | "
        f"train acc={train_accuracy * 100:.2f}% | "
        f"test loss={test_loss:.4f} | "
        f"test acc={test_accuracy * 100:.2f}%"
    )

Epoch 1/10: 100%|██████████| 196/196 [00:29<00:00,  6.57it/s, loss=0.9044]


Epoch 01/10 | lr=0.000976 | train loss=1.2203 | train acc=55.88% | test loss=1.0879 | test acc=62.31%


Epoch 2/10: 100%|██████████| 196/196 [00:29<00:00,  6.72it/s, loss=0.7721]


Epoch 02/10 | lr=0.000905 | train loss=0.7240 | train acc=74.19% | test loss=1.0771 | test acc=65.12%


Epoch 3/10: 100%|██████████| 196/196 [00:28<00:00,  6.86it/s, loss=0.6750]


...

Epoch 08/10 | lr=0.000095 | train loss=0.0055 | train acc=99.97% | test loss=0.5821 | test acc=85.19%


Epoch 9/10: 100%|██████████| 196/196 [00:28<00:00,  6.94it/s, loss=0.0028]


Epoch 09/10 | lr=0.000024 | train loss=0.0024 | train acc=100.00% | test loss=0.5820 | test acc=85.39%


Epoch 10/10: 100%|██████████| 196/196 [00:29<00:00,  6.73it/s, loss=0.0025]


Epoch 10/10 | lr=0.000000 | train loss=0.0020 | train acc=100.00% | test loss=0.5835 | test acc=85.34%


### Сравнение ResNet18 без дистилляции и с ней

In [ ]:
plain_loss, plain_accuracy = evaluate_student(student_plain, test_loader)
distilled_loss, distilled_accuracy = evaluate_student(student, test_loader)
teacher_loss, teacher_accuracy = evaluate_student(teacher, test_loader)

print(f"Resnet50: {teacher_accuracy * 100:.2f}")
print(f"ResNet18 без дистилляции: {plain_accuracy * 100:.2f}%")
print(f"ResNet18 с дистилляцией: {distilled_accuracy * 100:.2f}%")
print(f"Разница: {(distilled_accuracy - plain_accuracy) * 100:.2f} п.п.")

Resnet50: 94.39
ResNet18 без дистилляции: 85.34%
ResNet18 с дистилляцией: 87.91%
Разница: 2.57 п.п.
